# Claude Agent SDK · Python 从 0 到 1

> 一句话本质：**Client SDK 让你"调用模型"，Agent SDK 让你"雇一个会自己用工具干活的 Claude"**。前者你得自己写 tool loop，后者循环内建。

本笔记覆盖官方 Agent SDK 文档的全部主题（只讲 Python 版；TypeScript 仅在行为有差异时一句带过）。主线分四层递进：

1. **跑起来**（§3–§7）：安装认证 → 两个入口 → agent loop 原理 → 读消息流。
2. **控制它**（§8–§11）：配置总线 → 权限系统与 HITL → Hooks。
3. **扩展它**（§10、§12–§14）：自定义工具 → 外部 MCP → 会话 → 子 agent。
4. **上生产**（§15–§23）：多轮与打断 → 流式输入/输出 → 结构化输出 → 系统提示 → Claude Code 文件系统特性 → 生产化控制与安全部署。


## 全局记忆图

先把整张地图记住，后面每一节都是往这张图上填细节。主线是：**为什么 → 两个入口 → 一份配置 → 一条消息流 → 两个控制面 → 四类扩展 → 生产化**。

```mermaid
flowchart TD
    A["为什么用 Agent SDK<br/>(不想手写 tool loop)"] --> B{选入口}
    B -->|一次性任务| Q["query()<br/>发任务→收消息流"]
    B -->|多轮/可打断| C["ClaudeSDKClient<br/>持续会话"]
    Q --> O["ClaudeAgentOptions<br/>(唯一的配置总线)"]
    C --> O
    O --> S["消息流 async for<br/>System/Assistant/User/Result"]
    S --> BLK["内容块<br/>Text / Thinking / ToolUse / ToolResult"]
    S -.可选.-> SE["StreamEvent 逐 token 流式输出"]
    S -.可选.-> SO["structured_output 结构化输出"]

    O ==控制面==> P["权限系统<br/>hooks→deny→ask→mode→allow→回调"]
    O ==控制面==> H["Hooks<br/>生命周期拦截/审计/改写"]

    O -.扩展.-> T2["自定义工具<br/>@tool + 进程内 MCP"]
    O -.扩展.-> T3["外部 MCP<br/>stdio / http / sse"]
    O -.扩展.-> T4["会话<br/>continue / resume / fork"]
    O -.扩展.-> T5["子 agent<br/>AgentDefinition"]
    O -.扩展.-> T6["文件系统特性<br/>CLAUDE.md / skills / plugins"]

    P & H --> PROD["生产化<br/>预算 / sandbox / 可观测 / 安全部署"]

    style A fill:#fef3c7,stroke:#d97706
    style O fill:#dbeafe,stroke:#2563eb
    style S fill:#dcfce7,stroke:#16a34a
    style BLK fill:#dcfce7,stroke:#16a34a
    style P fill:#fee2e2,stroke:#dc2626
    style H fill:#fee2e2,stroke:#dc2626
    style PROD fill:#ede9fe,stroke:#7c3aed
```

四个颜色对应四个"必须先懂"的核心：黄=动机，蓝=配置总线（所有能力都从 `ClaudeAgentOptions` 挂进去），绿=消息流（agent 干活的过程全靠读这条流理解），红=控制面（权限与 hooks 决定 agent 能做什么）。


## 1–2. 架构定位与底层原理（已拆出独立笔记）

"为什么会有 Agent SDK""四个容易混的东西""同一内核两种外壳""子进程如何被启动和驱动"这四块内容已扩展成独立笔记：**[Claude-Agent-SDK-架构定位与子进程原理](./Claude-Agent-SDK-架构定位与子进程原理.md)**。

此处只留最小结论，保证本笔记可独立往下读：

- 分界线在**谁负责 tool loop**：Client SDK 手写循环，Agent SDK 循环内建——这是理解整个 SDK 的第一性原理。
- **CLI 面向人，SDK 面向程序**；Agent SDK 跑在应用自己的进程 + 本地文件系统上，Managed Agents 跑在 Anthropic 托管沙箱里。
- Agent SDK 与 Claude Code **共用同一个 agent 运行时**：SDK 安装包自带 CLI 二进制，运行时把它作为子进程拉起，通过 stdio 收发 newline-delimited JSON；agent loop、tool 执行、context 管理全在 CLI 子进程内，SDK 只是接口层。
- Python 侧靠一个事件循环（零额外线程）照看所有 agent 子进程；并发的瓶颈在内存（每个子进程是完整 Node 运行时），不在 CPU。

## 3. 安装与认证

- Python **≥ 3.10**（`No matching distribution found` 基本就是解释器太老）。
- SDK **自带 Claude Code CLI 二进制**（见[架构定位笔记](./Claude-Agent-SDK-架构定位与子进程原理.md)），`pip install claude-agent-sdk` 后即可运行，无需单独安装 Claude Code。CLI 查找顺序：`cli_path` 显式指定 > 包内捆绑二进制 > PATH 上的 `claude` 命令。

认证走环境变量，五选一（默认走 API key）：

| 提供方 | 环境变量 |
|---|---|
| Anthropic API（默认） | `ANTHROPIC_API_KEY` |
| Amazon Bedrock | `CLAUDE_CODE_USE_BEDROCK=1` + AWS 凭证 |
| Claude Platform on AWS | `CLAUDE_CODE_USE_ANTHROPIC_AWS=1` + `ANTHROPIC_AWS_WORKSPACE_ID` + AWS 凭证 |
| Google Vertex AI | `CLAUDE_CODE_USE_VERTEX=1` + GCP 凭证（展开见下） |
| Microsoft Azure | `CLAUDE_CODE_USE_FOUNDRY=1` + Azure 凭证 |

以 Vertex AI 为例，完整配置分两部分。第一部分是三个 Claude Code 侧变量：

```bash
export CLAUDE_CODE_USE_VERTEX=1                       # 启用 Vertex AI 通道
export CLOUD_ML_REGION=global                         # 端点位置：global / 多区域 us、eu / 具体区域如 us-east5
export ANTHROPIC_VERTEX_PROJECT_ID=YOUR-PROJECT-ID    # 用哪个 GCP 项目发请求
```

第二部分是 GCP 凭证本身——不走 Anthropic API key，走 GCP 标准的 Application Default Credentials（ADC），二选一：

- **本机开发**：`gcloud auth application-default login`，浏览器登录后凭证落盘，运行时自动读取，无需设任何变量。
- **服务器 / CI**：`export GOOGLE_APPLICATION_CREDENTIALS=/path/to/service-account-key.json`，指向 service account 密钥文件。

前提条件：GCP 项目已启用 Vertex AI API（`gcloud services enable aiplatform.googleapis.com`）、在 Model Garden 里申请开通目标 Claude 模型、账号具有 `roles/aiplatform.user` 角色。

> [!warning] 两个高频坑
> - **SDK 只从进程环境变量读 key，不会自动加载 `.env` 文件**。key 写在 `.env` 里时要自己先加载（如 `python-dotenv`），否则报 "API key not found"。
> - 官方明确：未经批准，第三方产品**不允许**用 claude.ai 登录态或其订阅额度来驱动基于 Agent SDK 的产品，请用上面的 API key 方式。


In [1]:
%pip install claude-agent-sdk

Note: you may need to restart the kernel to use updated packages.


## 4. 两个入口 × 两种输入模式

### 4.1 两个入口：`query()` 与 `ClaudeSDKClient`

| 维度 | `query()` | `ClaudeSDKClient` |
|---|---|---|
| 会话 | 每次调用**新开**一个 session | 复用同一 session |
| 对话 | 单次任务 | 同一上下文里连续追问 |
| 流式输入 | ✅（prompt 传 async generator） | ✅ |
| **打断 (interrupt)** | ❌ | ✅ |
| 续接上下文 | 手动（传 `resume` / `continue_conversation`） | 自动 |
| 适用 | 一次性任务、无状态环境（如 lambda） | 持续交互 / 需要中途打断 |

一句话：**能一句话说完的任务用 `query()`，要来回聊或要中途叫停用 `ClaudeSDKClient`**。

### 4.2 两种输入模式：单条消息 vs 流式输入

`prompt` 参数接受两种输入，对应官方说的两种输入模式：

- **单条消息（single message input）**：`prompt="..."` 传字符串。简单，但官方明确列出四个不支持：消息内附图、动态消息排队、实时打断、自然多轮对话。多轮只能靠 `continue_conversation=True` 或 `resume` 反复拉起。
- **流式输入（streaming input mode，官方推荐）**：`prompt` 传 async generator，逐条 yield 消息 dict；agent 作为长生命周期进程运行，支持附图、排队、打断、权限回调。`ClaudeSDKClient` 内部就是这种模式。详见 §16。

两者的共同点比差异更重要：**输入的是"任务"不是"问题"，输出的是"异步消息流"不是"一段字符串"**。这决定了 §7"怎么读消息流"是绕不过去的核心技能。

> [!warning] 单条消息模式的错误语义（高频坑）
> `query()` 以错误 result 收尾时（如 `error_max_turns`），SDK 会**先 yield 完最终 ResultMessage，然后 raise 一个普通 `Exception`**。凡是可能触限的 `async for` 循环都要包 `try/except`，否则后续代码不会执行。


## 5. 最小可运行示例：`query()`

第一次上手只开 `Read`，任务做成"读项目并总结"。先把消息流看懂，再谈写文件、跑 Bash。

- `prompt`：给 agent 一个**任务**，不是一句问答。
- `cwd`：Claude 站在哪个目录下做事；只要涉及读/改代码库就显式设。
- `allowed_tools`：**自动批准名单**（不是"全部可用工具列表"）。名单外的工具会走权限流程。
- `max_turns`：最多几轮，防任务无限延长，控成本。

工具组合直接决定 agent 能力档位：

| tools 组合 | agent 能做什么 |
|---|---|
| `Read`, `Glob`, `Grep` | 只读分析 |
| `Read`, `Edit`, `Glob` | 分析并修改代码 |
| `Read`, `Edit`, `Bash`, `Glob`, `Grep` | 完全自动化（改代码 + 跑测试） |


In [6]:
import anyio
from claude_agent_sdk import query, ClaudeAgentOptions


async def demo_query():
    options = ClaudeAgentOptions(
        cwd=".",
        max_turns=3,
        allowed_tools=["Read", "Glob"],  # 只自动批准读文件 / 找文件
    )
    async for message in query(
        prompt="Read the current project and summarize the main modules.",
        options=options,
    ):
        print(type(message).__name__, message)


# Jupyter 内核自带事件循环，直接 top-level await；脚本里改用 anyio.run(demo_query)
await demo_query()

SystemMessage SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/Users/liangzhu/Library/Mobile Documents/iCloud~md~obsidian/Documents/study-notes/AI/agents/claude-agent-sdk', 'session_id': 'd2acdb69-5f5d-42f3-bb06-478a711fa1a2', 'tools': ['Task', 'TaskOutput', 'Bash', 'Glob', 'Grep', 'ExitPlanMode', 'Read', 'Edit', 'Write', 'NotebookEdit', 'WebFetch', 'TodoWrite', 'WebSearch', 'KillShell', 'AskUserQuestion', 'Skill', 'EnterPlanMode', 'MCPSearch'], 'mcp_servers': [], 'model': 'claude-sonnet-4-5-20250929', 'permissionMode': 'default', 'slash_commands': ['compact', 'context', 'cost', 'init', 'pr-comments', 'release-notes', 'review', 'security-review'], 'apiKeySource': 'none', 'claude_code_version': '2.1.9', 'output_style': 'default', 'agents': ['Bash', 'general-purpose', 'statusline-setup', 'Explore', 'Plan'], 'skills': [], 'plugins': [], 'uuid': '0cdf3093-2668-488b-a166-6a797c34bc5b'})
AssistantMessage AssistantMessage(content=[TextBlock(text="I'll explore 

## 6. Agent loop 工作原理

上一节跑通了，这一节讲清楚 `async for` 背后那个循环在做什么。

### 6.1 循环五步

1. **收 prompt**：Claude 收到 prompt + system prompt + tool 定义 + 对话历史。SDK yield 一条 `subtype="init"` 的 `SystemMessage`（含 session 元数据）。
2. **评估并响应**：Claude 可能回文本、请求一个或多个 tool call、或两者都有。SDK yield `AssistantMessage`。
3. **执行 tools**：SDK 执行每个请求的 tool 并把结果回灌给 Claude（以 `UserMessage` 形式出现在流里）。hooks 可以在执行前拦截/修改/阻断。
4. **重复**：步骤 2–3 循环。**一个完整周期 = 一个 turn**；`max_turns` 只计带 tool call 的 turn。
5. **收尾**：Claude 产出不含 tool call 的纯文本响应后，SDK yield 最终 `AssistantMessage`，随后是 `ResultMessage`（最终文本、token 用量、费用、session ID）。

简单问题 1–2 turn；复杂任务可跨几十个 tool call。**开放式 prompt（"improve this codebase"）可能跑很久，生产 agent 默认就该设 `max_turns` / `max_budget_usd`。**

### 6.2 内置 tools 全景

| 类别 | Tools | 作用 |
|---|---|---|
| 文件操作 | `Read`, `Edit`, `Write` | 读、改、建文件 |
| 搜索 | `Glob`, `Grep` | 按模式找文件、正则搜内容 |
| 执行 | `Bash` | shell 命令、脚本、git |
| Web | `WebSearch`, `WebFetch` | 搜网、抓取解析页面 |
| 发现 | `ToolSearch` | 按需动态查找加载 tools（见 §12.4） |
| 编排 | `Agent`, `Skill`, `AskUserQuestion`, `TaskCreate`, `TaskUpdate` | 派子 agent、调 skill、问用户、跟踪任务 |

并行规则：一个 turn 内的多个 tool call，只读 tools（`Read`/`Glob`/`Grep`/标记 read-only 的 MCP tools）可并发；改状态的 tools（`Edit`/`Write`/`Bash`）串行。**自定义 tool 默认按串行处理，要参与并行需在 annotations 里标 `readOnlyHint=True`（见 §10.3）。**

### 6.3 Context window：什么在吃你的上下文

session 内 context **不会在 turn 之间重置**，一切累积。各项来源：

| 来源 | 何时加载 | 影响 |
|---|---|---|
| System prompt | 每次请求 | 小的固定开销（prompt cache 命中） |
| CLAUDE.md | session 开始，经 `setting_sources` | 全文进每次请求 |
| Tool 定义 | 每次请求；MCP schema 默认延迟加载 | 内置 tool schema 每次都在；tool search 把 MCP schema 挡在外面 |
| 对话历史 | 随 turn 累积 | 每 turn 增长；大文件读取单次就能吃掉数千 token |
| Skill 描述 | session 开始 | 只有简短摘要，调用时才加载全文 |

context 接近上限时 SDK **自动 compaction**：把旧历史总结压缩，流里出现 `subtype="compact_boundary"` 的 `SystemMessage`。

> [!warning] Compaction 会丢早期指令
> compaction 用摘要替换旧消息，对话早期的具体指令可能丢失。**必须持久生效的规则放 CLAUDE.md**（每次请求重新注入，不受 compaction 影响）。可在 CLAUDE.md 写一节 "Summary instructions" 指定压缩时必须保留什么；`PreCompact` hook 可在压缩前归档全文；把 `/compact` 作为 prompt 发送可手动触发压缩。

保持 context 高效的四个手段：子任务交给 subagent（父级只收最终摘要，见 §14）、按 `AgentDefinition.tools` 精简工具集、留意 MCP schema 成本、常规任务用低 `effort`。


## 7. 核心技能：怎么读消息流

新手最容易卡的不是发任务，而是**看不懂 `async for` 里飞出来的一堆对象**。消息流分两层：外层是 **Message**（一条消息），内层是 **ContentBlock**（消息里的内容块）。Python 里全部用 `isinstance()` 分流，所有消息类（含 `StreamEvent`、`RateLimitEvent` 和 SystemMessage 的具名子类）都从 `claude_agent_sdk` 顶层导入。

类型系统只有两条组织规则：**顶层六种主类型对应 wire 上的六种 `type` 字段**；**`type == "system"` 的消息再按 `subtype` 细分**——高频 subtype 被 SDK 升格成 `SystemMessage` 的具名子类（hook 事件、后台任务生命周期等），其余走通用 `SystemMessage`。SDK 对不认识的消息类型直接跳过（新 CLI 配旧 SDK 不会崩），所以处理代码要容忍"流里出现没见过的东西"。

### 7.1 Message 六种主类型

| 类型 | 含义 | 关键字段 |
|---|---|---|
| `SystemMessage` | 系统事件（含一组具名子类，见 §7.2） | `subtype`、`data` |
| `AssistantMessage` | Claude 的一轮输出 | `content`（一串 ContentBlock）、`model`、`parent_tool_use_id`、`error`、`usage`、`message_id`、`stop_reason`、`session_id` |
| `UserMessage` | 用户输入 / tool 结果回填 | `content`、`parent_tool_use_id`、`tool_use_result` |
| `StreamEvent` | 逐 token 增量事件（仅开启 `include_partial_messages` 时出现） | `event`（原始 API 事件 dict），见 §17 |
| `RateLimitEvent` | 限流状态变化时推送 | `rate_limit_info`：`status`（`allowed` / `allowed_warning` / `rejected`）、`resets_at`、`utilization`、`rate_limit_type` |
| `ResultMessage` | 本次任务**收尾** | `result`、`subtype`、`is_error`、`num_turns`、`session_id`、`total_cost_usd`、`usage`、`model_usage`、`structured_output`、`stop_reason`、`permission_denials`、`api_error_status` |

- `AssistantMessage.error` 是本轮失败的分类标签：`authentication_failed` / `billing_error` / `rate_limit` / `invalid_request` / `server_error` / `unknown`。
- `RateLimitEvent` 用于限流预警：`allowed_warning` 时提示用户接近限额，`rejected` 时主动退避到 `resets_at`。

### 7.2 SystemMessage 家族：通用 subtype + 六个具名子类

通用 `SystemMessage` 的常见 `subtype`：`"init"`（首条，`data["session_id"]` 是会话 ID，`data` 里还有 tools / mcp_servers / slash_commands / skills / plugins 清单）、`"compact_boundary"`（发生了 compaction）、`"informational"`（状态横幅）。

六个高频 subtype 有具名子类，字段直接是属性、不用翻 `data` dict：

| 子类 | 何时出现 | 关键字段 |
|---|---|---|
| `HookEventMessage` | hook 开始/结束（需 `include_hook_events=True`；subtype 为 `hook_started` / `hook_response`） | `hook_event_name`（如 `"PreToolUse"`）；`hook_response` 的 `data` 里有 `output` / `exit_code` / `outcome` |
| `TaskStartedMessage` | 后台任务启动 | `task_id`、`description`、`task_type`、`tool_use_id` |
| `TaskProgressMessage` | 后台任务运行中定期上报 | `task_id`、`usage`（`total_tokens` / `tool_uses` / `duration_ms`）、`last_tool_name` |
| `TaskNotificationMessage` | 后台任务完成 / 失败 / 被停止 | `task_id`、`status`（`completed` / `failed` / `stopped`）、`summary`、`output_file`、`usage` |
| `TaskUpdatedMessage` | 后台任务状态变更 | `task_id`、`patch`（变更字段）、`status`（`pending` / `running` / `paused` / `completed` / `failed` / `killed`） |
| `MirrorErrorMessage` | `session_store` 镜像写失败（SDK 合成，见 §13.3） | `key`、`error` |

"任务（task）"指**后台运行的工作**：后台派发的子 agent（`AgentDefinition(background=True)` 或默认后台化的派发）、`run_in_background` 的 Bash 命令。用 `task_id` 定位，`client.stop_task(task_id)` 可停止。

两个使用要点：

- **所有子类都继承 `SystemMessage`**，`isinstance(msg, SystemMessage)` 照样命中，旧代码不破。反过来，if/elif 链要细分时必须**把子类判断放在 `SystemMessage` 之前**，否则永远落进父类分支。
- **后台任务的终态可能只以 `TaskUpdatedMessage` 出现**（`patch.status` 进入终态），不一定伴随 `TaskNotificationMessage`——比如 `stop_task()` 杀掉的任务往往只发一条 `status="killed"` 的 update。跟踪活跃任务要在**两种消息**的终态上都做清理；终态集合可直接导入：`TERMINAL_TASK_STATUSES = {"completed", "failed", "stopped", "killed"}`。

### 7.3 ResultMessage 的 subtype 全表

| subtype | 含义 | `result` 是否可用 |
|---|---|---|
| `success` | 正常完成 | ✅ |
| `error_max_turns` | 触到 `max_turns` | ❌ |
| `error_max_budget_usd` | 触到 `max_budget_usd` | ❌ |
| `error_during_execution` | 执行中断（API 失败、请求取消等） | ❌ |
| `error_max_structured_output_retries` | 结构化输出重试耗尽（见 §18） | ❌ |

四个配套的坑：

- **`result` 只在 `success` 上存在**，读之前必须先查 subtype；所有 subtype 都带 `session_id`（错误后也能 resume）。
- `total_cost_usd` / `usage` 类型是 optional，某些错误路径上是 `None`，格式化前先判空。
- **ResultMessage 之后可能还有 trailing 事件**（如 prompt 建议），要把流迭代到自然结束，不要收到 result 就 `break`。
- `api_error_status` 在 API 调用失败时记录 HTTP 状态码（429 / 500 / 529），不含消息内容，可直接进日志和告警。

`stop_reason`（`str | None`）记录模型最后一轮为何停止：`end_turn`（正常）、`max_tokens`、`refusal`（可用它检测模型拒绝）。

### 7.4 ContentBlock 四种块（在 `AssistantMessage.content` 里）

| 块 | 含义 |
|---|---|
| `TextBlock` | 普通文本（`.text`） |
| `ThinkingBlock` | 扩展思考内容（`.thinking`、`.signature`） |
| `ToolUseBlock` | Claude 要调某工具（`.name` / `.input` / `.id`） |
| `ToolResultBlock` | 工具返回结果（`.tool_use_id` / `.content` / `.is_error`） |

另有两个 server 端块偶尔出现：`ServerToolUseBlock` / `ServerToolResultBlock`（服务端执行的工具，如 advisor tool 的调用与结果）。做通用渲染时给一个兜底分支即可。

**心智模型**：一条 `AssistantMessage` 可能同时含"思考 + 文字 + 要调工具"。按块类型 `isinstance` 分流，就能重建出 agent 的完整行为轨迹（想什么→说什么→调什么→拿到什么）。按需求选处理层级：只要结果读 `ResultMessage`；要进度读 `AssistantMessage`（后台任务的进度另看 §7.2 的 Task 系消息）；要打字机效果开 `include_partial_messages` 读 `StreamEvent`（§17）。


In [ ]:
import anyio
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    SystemMessage,
    AssistantMessage,
    ResultMessage,
    TextBlock,
    ThinkingBlock,
    ToolUseBlock,
    ToolResultBlock,
)


async def demo_read_stream():
    session_id = None
    async for msg in query(
        prompt="List the files here, then summarize the project in one sentence.",
        options=ClaudeAgentOptions(
            cwd=".", allowed_tools=["Bash", "Glob", "Read"], max_turns=4
        ),
    ):
        if isinstance(msg, SystemMessage) and msg.subtype == "init":
            session_id = msg.data["session_id"]  # 记下会话 ID，§13 resume 要用
            print("session:", session_id)
        elif isinstance(msg, AssistantMessage):
            for b in msg.content:
                if isinstance(b, ThinkingBlock):
                    print("[thinking]", b.thinking[:80])
                elif isinstance(b, TextBlock):
                    print("[text]", b.text)
                elif isinstance(b, ToolUseBlock):
                    print("[tool_use]", b.name, b.input)
                elif isinstance(b, ToolResultBlock):
                    print("[tool_result]", str(b.content)[:80])
        elif isinstance(msg, ResultMessage):
            print(
                "[done]",
                "ok" if not msg.is_error else "ERROR",
                "turns=",
                msg.num_turns,
                "cost$=",
                msg.total_cost_usd,
            )
            print("final:", msg.result if msg.subtype == "success" else msg.subtype)


await demo_read_stream()

## 8. `ClaudeAgentOptions`：唯一的配置总线

所有能力都从这个 dataclass 挂进去。不用背全部字段，按**七组功能**记（每组括注本笔记展开的章节）：

**① 工具与权限**（§9、§10）
- `allowed_tools: list[str]`：自动批准名单
- `disallowed_tools: list[str]`：黑名单（裸名=从 context 移除；带范围如 `"Bash(rm *)"`=只 deny 匹配调用）
- `tools: list[str] | preset`：**可用性名单**——只有列出的内置 tool 进 context（与 allowed_tools 是两层概念，见 §10.4）
- `permission_mode` / `can_use_tool` / `permission_prompt_tool_name`：权限模式 / 审批回调 / 自定义审批 MCP tool
- `sandbox: SandboxSettings`：命令沙箱（§21.5）

**② 系统提示与模型**（§19）
- `system_prompt`：字符串，或 preset dict `{"type": "preset", "preset": "claude_code", "append": "..."}`；**不设时用的是只覆盖 tool calling 的最小 prompt，不是 Claude Code 完整人格**
- `model` / `fallback_model`：模型 ID / 主模型失败时的兜底
- `effort`：`"low" | "medium" | "high" | "xhigh" | "max"`，用延迟和 token 换推理深度
- `thinking: ThinkingConfig`：extended thinking 配置（与 effort 正交；旧字段 `max_thinking_tokens` 已废弃）

**③ 会话**（§13）
- `resume` / `continue_conversation` / `fork_session`：恢复指定 session / 接最近一次 / 分叉
- `session_store` / `session_store_flush`：外部存储 adapter / 刷写策略（`"batched"` 每 turn 一次，`"eager"` 每帧）
- `enable_file_checkpointing`：文件改动快照，可回滚（§21.4）

**④ 运行环境**
- `cwd` / `add_dirs`：工作目录 / 额外可访问目录
- `env`：合并进 CLI 子进程的环境变量（`ENABLE_TOOL_SEARCH`、`CLAUDE_CONFIG_DIR`、OTel、超时重试等开关都从这里进）
- `setting_sources`：文件系统配置来源；**省略 = `["user", "project", "local"]` 全加载**，`[]` = 只认代码配置（§20）
- `settings` / `cli_path` / `extra_args` / `max_buffer_size` / `stderr` / `user`：settings 文件路径 / CLI 路径 / 透传 CLI 参数 / stdout 缓冲上限 / stderr 回调 / 用户标识

**⑤ 扩展**
- `mcp_servers`：外部/进程内 MCP（§10、§12）；`strict_mcp_config=True` 时只认这里传的，无视 `.mcp.json`、用户配置和 claude.ai connectors
- `hooks`：生命周期钩子（§11）
- `agents: dict[str, AgentDefinition]`：子 agent（§14）
- `skills` / `plugins`：技能过滤 / 插件加载（§20）
- `betas: list[SdkBeta]`：beta 特性开关

**⑥ 输出形态**
- `include_partial_messages`：开启后流里多出 `StreamEvent`（§17）
- `output_format`：`{"type": "json_schema", "schema": {...}}` 结构化输出（§18）
- `include_hook_events`：把 hook 生命周期事件透出到消息流

**⑦ 预算与轮次**
- `max_turns`：最多 tool-use 轮次
- `max_budget_usd`：客户端成本估算达到该美元数即停（与 `total_cost_usd` 同一口径，见 §21.2）

> 💡 `allowed_tools` 是"自动批准"而非"能用的全集"——不在名单里的工具**不是不能用，是会触发审批**；真正控制"哪些工具存在"的是 `tools` 字段。这组概念分不清是新手第一大坑，§9–§10 展开。


## 9. 权限系统与 Human-in-the-Loop

### 9.1 权限评估顺序（先记这条流水线）

Claude 每请求一次 tool，SDK 按固定顺序评估，**前面的步骤解决了，后面的就不再执行**：

```text
Hooks → Deny 规则 → Ask 规则 → Permission mode → Allow 规则 → can_use_tool 回调
```

1. **Hooks** 最先跑。hook 可以直接 deny；但 hook 返回 allow **不会跳过**后面的 deny/ask 规则。
2. **Deny 规则**（`disallowed_tools` + settings.json）匹配即阻断，**连 `bypassPermissions` 都拦得住**。
3. **Ask 规则** 匹配则强制落入 `can_use_tool` 询问；`AskUserQuestion` 等"必须与人交互"的 tool 也总是落到回调。
4. **Permission mode**：`bypassPermissions` 批准到这一步的一切；`acceptEdits` 批准文件操作；`plan` 把写操作强制引回回调（无视 allow 规则）。
5. **Allow 规则**（`allowed_tools` + settings.json）匹配即批准。
6. **`can_use_tool` 回调** 兜底；`dontAsk` 模式下跳过此步直接 deny。

由这条流水线推出三个关键结论：

- **被前面任何一步批准的调用，永远到不了 `can_use_tool`**——写在回调里的检查对 `allowed_tools` 里的 tool 静默失效。必须每次调用都生效的逻辑，用 `PreToolUse` hook（它在整条流水线之前，连 `bypassPermissions` 都绕不过它的 deny）。
- **`allowed_tools` 不约束 `bypassPermissions`**：`allowed_tools=["Read"]` + `bypassPermissions` 仍然批准所有工具。要在该模式下禁工具只能用 `disallowed_tools`。
- **deny 规则两种写法语义不同**：裸名 `"Bash"` 把 tool 定义整个从 context 移除（Claude 根本看不见）；带范围 `"Bash(rm *)"` 保留 tool、只 deny 匹配的调用。

### 9.2 permission_mode 全表

| 模式 | 行为 | 适用 |
|---|---|---|
| `default` | 未被规则覆盖的 tool 触发 `can_use_tool`；没配回调则 deny | 交互应用 + 审批回调 |
| `acceptEdits` | 自动批准文件编辑与文件系统命令（`mkdir`/`touch`/`rm`/`mv`/`cp`/`sed`），仅限 `cwd` 和 `add_dirs` 范围内 | 受信任的开发工作流 |
| `plan` | 只探索不改动；写操作永不自动批准，经回调询问 | 先规划再动手 |
| `dontAsk` | 从不询问：预批准的运行，其余直接 deny，回调不会被调用 | 锁死的 headless agent |
| `bypassPermissions` | 一律放行（除 deny/ask 规则和 hooks）；Unix root 下不可用 | 沙箱 CI、隔离环境 |

（TypeScript 另有模型分类器审批的 `auto` 模式，Python 无。）

推荐配对：交互应用 `default` + 回调；开发机自主 agent `acceptEdits`；锁死型 agent `allowed_tools` + `dontAsk`；`bypassPermissions` 只留给容器/CI。**MCP tool 的授权优先用 `allowed_tools` 通配符（如 `mcp__github__*`），不要靠放宽 permission mode**——`acceptEdits` 不批 MCP tool，`bypassPermissions` 又批得太宽。

运行期可切模式：`await client.set_permission_mode("acceptEdits")`（`ClaudeSDKClient`），典型用法是先严后松。

### 9.3 `can_use_tool` 回调（HITL 的落点）

回调签名与返回值：

```python
async def can_use_tool(
    tool_name: str,                    # "Bash" / "Write" / "AskUserQuestion"...
    input_data: dict,                  # tool 入参，内容随 tool 而异
    context: ToolPermissionContext,    # 携带 suggestions（现成的权限更新建议）
) -> PermissionResultAllow | PermissionResultDeny: ...
```

- `PermissionResultAllow(updated_input=...)`：放行，还能**改写工具入参**（比如把写路径重定向到沙箱；Claude 不知道被改过）。
- `PermissionResultAllow(updated_input=..., updated_permissions=...)`：批准并持久化规则（从 `context.suggestions` 里挑 `destination == "localSettings"` 的条目回传，即"总是允许"；需 SDK ≥ 0.1.80）。
- `PermissionResultDeny(message=..., interrupt=...)`：拒绝并给模型一个理由——理由写得好，Claude 会换方案（如"用户不想删文件，问能否改成压缩归档"）。

因为回调是 `async`，可以在里面 `await` 一个 Future 挂起，等真人点"批准/拒绝"再返回——把"异步等人"伪装成一次同步权限判断，Human-in-the-Loop 的本质就是这个。回调可以无限期 pending；等待时长可能超过进程存活时，改用 hook 的 `defer` 决策先退出、之后 resume。

> [!warning] Python 专属坑：回调需要流式输入 + dummy hook
> Python 里 `can_use_tool` 要求 **streaming mode**（`prompt` 传 async generator，或直接用 `ClaudeSDKClient`）。用 `query()` 时还需注册一个返回 `{"continue_": True}` 的 `PreToolUse` hook 保持流打开（注意键名带下划线），否则流会在回调触发前关闭。下面的示例即官方推荐写法。


In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
from claude_agent_sdk.types import (
    HookMatcher,
    PermissionResultAllow,
    PermissionResultDeny,
    ToolPermissionContext,
)


async def gate(tool_name: str, input_data: dict, context: ToolPermissionContext):
    # 例1：禁止危险删除，并给模型一个可行的替代方向
    if tool_name == "Bash" and "rm " in input_data.get("command", ""):
        return PermissionResultDeny(
            message="User doesn't want to delete files; compress them into an archive instead."
        )
    # 例2：把碰 config 的写操作重定向到沙箱（改写入参后放行，Claude 无感知）
    if tool_name in ("Write", "Edit") and "config" in input_data.get("file_path", ""):
        safe = f"./sandbox/{input_data['file_path']}"
        return PermissionResultAllow(updated_input={**input_data, "file_path": safe})
    # 例3（HITL 骨架）：真人审批
    #   fut = asyncio.get_running_loop().create_future()
    #   push_to_ui(tool_name, input_data, fut)   # 前端弹审批框，点击后 fut.set_result(bool)
    #   return PermissionResultAllow(updated_input=input_data) if await fut \
    #       else PermissionResultDeny(message="user denied")
    return PermissionResultAllow(updated_input=input_data)


# Python 必需的 workaround：dummy PreToolUse hook 保持流打开，否则回调不会被触发
async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}


# can_use_tool 要求流式输入：prompt 传 async generator 而非字符串
async def prompt_stream():
    yield {
        "type": "user",
        "message": {"role": "user", "content": "Update the app config file"},
    }


async def demo_gate():
    async for m in query(
        prompt=prompt_stream(),
        options=ClaudeAgentOptions(
            can_use_tool=gate,
            hooks={"PreToolUse": [HookMatcher(matcher=None, hooks=[dummy_hook])]},
        ),
    ):
        if isinstance(m, ResultMessage) and m.subtype == "success":
            print(m.result)


await demo_gate()

### 9.4 另一条 HITL 通道：`AskUserQuestion`

上面的 `can_use_tool` 是**你拦 Claude**；`AskUserQuestion` 是**Claude 主动问你**。它是个内建工具，Claude 觉得多个方案都合理、需要方向时调用（`plan` 模式下尤其常见）。它同样触发 `can_use_tool` 回调（`tool_name == "AskUserQuestion"`），你负责把问题呈现给真人、把选择回填——**问题和选项由 Claude 生成，你不能往这个流程里塞自己的问题**。

**输入格式**（`input_data["questions"]`，每次 1–4 个问题）：

| 字段 | 含义 |
|---|---|
| `question` | 完整问题文本 |
| `header` | 短标签（≤12 字符） |
| `options` | 2–4 个选项，各有 `label` 和 `description` |
| `multiSelect` | `true` 则可多选 |

**回填格式**：返回 `PermissionResultAllow(updated_input=...)`，其中 `updated_input` 必须包含**原样传回的 `questions`** 加上 `answers`——key 是问题文本，value 是所选 `label`（多选传 label 列表）。用户打了自由文本就直接放原文，不要放 "Other"：

```python
return PermissionResultAllow(
    updated_input={
        "questions": input_data.get("questions", []),
        "answers": {
            "How should I format the output?": "Summary",
            "Which sections should I include?": ["Introduction", "Conclusion"],
        },
    }
)
```

两个限制：subagent 内当前不可用 `AskUserQuestion`；如果用 `tools` 字段收窄了工具集，必须把 `"AskUserQuestion"` 加回名单，否则 Claude 无法提问。

两条通道最终都汇成"agent 停下等人输入"的状态。更复杂的交互（表单、多步向导、对接外部审批系统）用自定义工具实现（§10），那是控制力最强、实现成本也最高的一档。


## 10. 自定义工具：进程内 MCP Server

让 Claude 调你自己的本地能力（业务函数、数据库封装、配置中心），是 Agent SDK 真正产生工程价值的地方。做法是把 Python 函数包成**进程内 SDK MCP Server**——不用单独起外部服务，工具和你的应用同进程。

### 10.1 三步接入

1. `@tool(name, description, input_schema)` 把 async 函数声明成工具，handler 返回 `{"content": [{"type": "text", "text": ...}]}`。
2. `create_sdk_mcp_server(name, version, tools=[...])` 打包成 server 对象。
3. 挂到 `mcp_servers={...}`，并把工具全名加进 `allowed_tools`（**MCP tool 必须显式授权，否则 Claude 看得到调不了**）。

工具全名规则：`mcp__<server名>__<tool名>`，server 名取 `mcp_servers` dict 的 key。同一 server 的全部工具可用通配符 `mcp__<server名>__*`。

### 10.2 Schema 的两档写法

- **简易 dict**：`{"latitude": float}`，SDK 自动转 JSON Schema。**每个 key 都视为必填**；要可选参数就不写进 schema，在 description 里说明、handler 里用 `args.get("hours", 12)` 读。
- **完整 JSON Schema dict**：需要 `enum`、取值范围、嵌套对象时用，`@tool` 直接接受（`{"type": "object", "properties": {...}, "required": [...]}`）。

### 10.3 Annotations 与并行

`@tool(..., annotations=ToolAnnotations(readOnlyHint=True))`。`readOnlyHint` 是唯一有行为影响的 hint——标了它，Claude 才会把这个工具与其他只读工具并行调用；其余 hint（`destructiveHint`/`idempotentHint`/`openWorldHint`）仅供参考。**annotation 是元数据不是约束**，标了只读的 handler 照样能写盘，自己保持一致。

### 10.4 `tools` vs `allowed_tools`：可用性层 vs 权限层

| 选项 | 作用层 | 效果 |
|---|---|---|
| `tools=["Read", "Grep"]` | 可用性 | 只有列出的**内置** tool 进 context；MCP tool 不受影响 |
| `tools=[]` | 可用性 | 移除全部内置 tool，Claude 只剩你的 MCP tool |
| `allowed_tools=[...]` | 权限 | 列出的免审批；未列出的仍可用，走权限流程 |
| `disallowed_tools=["Bash"]`（裸名） | 可用性 | 从 context 移除，Claude 不会尝试 |
| `disallowed_tools=["Bash(rm *)"]`（带范围） | 权限 | tool 仍可见，只 deny 匹配调用（Claude 可能浪费一轮去试） |

### 10.5 错误处理铁律

| handler 行为 | 后果 |
|---|---|
| 抛出未捕获异常 | **整个 agent loop 终止**，Claude 看不到错误 |
| 捕获后返回 `{"content": [...], "is_error": True}` | 循环继续，Claude 把错误当数据，可重试/换工具/解释失败 |

> [!note] 其他返回形态与 Python 限制
> `content` 数组还接受 `image` block（`data` 放纯 base64 字节，无 `data:` 前缀，`mimeType` 必填）和 `resource` block（URI 只是引用标签，内容在 `text`/`blob` 里）。`structuredContent`（机器可读 JSON 结果）**Python 进程内 server 不支持**——`@tool` 只转发 `content` 和 `is_error`，需要它就得跑独立 MCP server。


In [ ]:
from claude_agent_sdk import (
    tool,
    create_sdk_mcp_server,
    ClaudeAgentOptions,
    ClaudeSDKClient,
)


@tool("greet", "Greet a user", {"name": str})
async def greet_user(args):
    return {"content": [{"type": "text", "text": f"Hello, {args['name']}!"}]}


@tool("add", "Add two numbers", {"a": float, "b": float})
async def add(args):
    return {"content": [{"type": "text", "text": f"Sum: {args['a'] + args['b']}"}]}


# 错误处理铁律的落地：捕获异常并返回 is_error，循环不中断，Claude 能看到并应对
@tool("divide", "Divide a by b", {"a": float, "b": float})
async def divide(args):
    try:
        return {
            "content": [{"type": "text", "text": f"Quotient: {args['a'] / args['b']}"}]
        }
    except ZeroDivisionError:
        return {
            "content": [{"type": "text", "text": "Error: division by zero"}],
            "is_error": True,
        }


server = create_sdk_mcp_server(
    name="demo", version="1.0.0", tools=[greet_user, add, divide]
)


async def demo_custom_tool():
    options = ClaudeAgentOptions(
        mcp_servers={"demo": server},
        allowed_tools=["mcp__demo__*"],  # 通配符批准 demo server 的全部工具
        max_turns=3,
    )
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Greet Liangzhu, then add 2 and 3, then divide 1 by 0.")
        async for m in client.receive_response():
            print(m)


await demo_custom_tool()

## 11. Hooks：把 agent 变成可控系统

Hook **不是给 Claude 用的工具**，而是给"包在 agent 循环外面的应用控制层"用的回调：在生命周期的确定时机跑你的代码，用来拦截危险操作、审计合规、改写输入输出、脱敏、注入 context。Hooks 跑在你的应用进程里、不占 context window；`PreToolUse` 的 deny 是唯一连 `bypassPermissions` 都拦得住的机制（见 §9.1）。

### 11.1 Python 可用的 hook 事件（10 个）

| 事件 | 触发时机 | 典型用途 |
|---|---|---|
| `PreToolUse` | tool 执行前（可阻断/改写） | 拦危险命令、重定向路径 |
| `PostToolUse` | tool 返回后 | 审计日志、改写 tool 输出（`updatedToolOutput`） |
| `PostToolUseFailure` | tool 执行失败 | 记录/处理 tool 错误 |
| `UserPromptSubmit` | prompt 提交时 | 注入额外 context（`additionalContext`） |
| `Stop` | agent 执行停止 | 保存状态（忽略 matcher） |
| `SubagentStart` / `SubagentStop` | 子 agent 启动/完成 | 跟踪并行任务 |
| `PreCompact` | compaction 前 | 归档完整 transcript |
| `PermissionRequest` | 将弹权限询问时 | 发外部通知（Slack/邮件） |
| `Notification` | agent 状态消息 | 转发状态到监控系统 |

（`SessionStart`/`SessionEnd`/`PostToolBatch`/`MessageDisplay` 等仅 TypeScript 有；Python 想跑 session 级 hook 只能用 settings.json 里的 shell command hook，经 `setting_sources=["project"]` 加载。）

### 11.2 配置结构与 matcher 规则

```python
hooks={"PreToolUse": [HookMatcher(matcher="Write|Edit", hooks=[callback], timeout=60)]}
```

matcher 匹配规则（大小写敏感，只匹配 tool 名，不匹配文件路径等参数）：

- 只含字母数字、`_`、`-`、空格、`,`、`|` → **精确匹配**，`|` 或 `,` 分隔多候选（`"Write|Edit"`）。
- `*`、空串、省略 → 匹配该事件全部发生。
- 含其他字符 → 按**未锚定正则**求值（`"^mcp__"` 匹配所有 MCP tool；`"Edit.*"` 同时命中 `Edit` 和 `NotebookEdit`）。

> [!warning] MCP matcher 高频坑
> `matcher="mcp__memory"` 落在精确匹配字符集里，只会精确比较、**匹配不到任何 tool**。要匹配某 server 全部工具写 `mcp__memory__.*`。

### 11.3 回调签名与返回值

```python
async def my_hook(input_data: dict, tool_use_id: str | None, context) -> dict: ...
```

`input_data` 随事件而异（`PreToolUse` 有 `tool_name`/`tool_input`；全部事件共享 `session_id`/`cwd`/`hook_event_name`）。返回 `{}` = 放行。要干预就返回：

```python
{
    "systemMessage": "给用户看的提示（可选）",
    "continue_": True,          # 顶层字段；Python 用下划线避开关键字
    "hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "allow" | "deny" | "ask" | "defer",
        "permissionDecisionReason": "给模型看的原因",
        "updatedInput": {...},  # 改写入参；必须配 allow 或 ask 才生效
    },
}
```

多个 hook 命中同一事件时**并行执行**，权限决策取最严：`deny > defer > ask > allow`。只做副作用（日志/webhook）不想拖慢 agent 时，返回 `{"async_": True}` 让 agent 立即继续（async 输出不能再阻断或改写）。

**Hook vs `can_use_tool` 怎么选**：审批交互（等真人点头）用 `can_use_tool`；必须每次调用都生效的确定性检查、审计、多生命周期点干预用 Hooks。

> [!note] 排障速查
> hook 不触发：事件名大小写、matcher 精确度、`max_turns` 触限时 hook 可能来不及跑。改写不生效：`updatedInput` 必须在 `hookSpecificOutput` 内且带 `permissionDecision: "allow"`。hook 内抛异常会打断 agent——发 HTTP 等副作用要自己 try/except，阻塞调用用 `asyncio.to_thread` 包。


In [ ]:
from datetime import datetime
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher, ResultMessage


# PreToolUse：阻断对 .env 的任何写入，reason 给模型、systemMessage 给用户
async def protect_env_files(input_data, tool_use_id, context):
    file_path = input_data["tool_input"].get("file_path", "")
    if file_path.split("/")[-1] == ".env":
        return {
            "systemMessage": ".env is protected.",
            "hookSpecificOutput": {
                "hookEventName": input_data["hook_event_name"],
                "permissionDecision": "deny",
                "permissionDecisionReason": "Cannot modify .env files",
            },
        }
    return {}


# PostToolUse：把所有文件改动写进审计日志
async def log_file_change(input_data, tool_use_id, context):
    fp = input_data.get("tool_input", {}).get("file_path", "unknown")
    with open("./audit.log", "a") as f:
        f.write(f"{datetime.now()}: modified {fp}\n")
    return {}  # 返回空 dict = 放行


async def demo_hook():
    async for message in query(
        prompt="Refactor utils.py to improve readability",
        options=ClaudeAgentOptions(
            permission_mode="acceptEdits",
            hooks={
                "PreToolUse": [
                    HookMatcher(matcher="Write|Edit", hooks=[protect_env_files])
                ],
                "PostToolUse": [
                    HookMatcher(matcher="Edit|Write", hooks=[log_file_change])
                ],
            },
        ),
    ):
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_hook()

## 12. 外部 MCP：接入现成生态

除了进程内自定义工具，还能连**外部 MCP server**（数据库、浏览器、GitHub、Slack，社区有几百个现成的）。

### 12.1 三种 transport（选型规则：文档给命令 → stdio；给 URL → http/sse）

| 类型 | 配置形状 | 说明 |
|---|---|---|
| `stdio` | `{"command": "npx", "args": [...], "env": {...}}` | 本地子进程，`env` 传凭证 |
| `http` | `{"type": "http", "url": "...", "headers": {...}}` | 远程 server，`headers` 传认证头；代码配置只接受 `"http"`（`.mcp.json` 里 `"streamable-http"` 是别名） |
| `sse` | `{"type": "sse", "url": "...", "headers": {...}}` | 远程 SSE server |

配置有两条途径：代码里传 `mcp_servers={...}`，或项目根放 `.mcp.json`（`setting_sources` 含 `"project"` 时自动读取，`${VAR}` 语法运行时展开环境变量）。

### 12.2 授权与发现

- **MCP tool 必须显式加进 `allowed_tools` 才能被调用**，没授权时 Claude 看得到但调不了（"工具明明连上了却不用"多半是这个原因）。通配符只能作用在 tool 段：`mcp__github__*` 有效，`mcp__*` 会被忽略并告警。
- 发现 server 提供了什么 tool：读 init 消息的 `message.data["mcp_servers"]`，同时检查每个 server 的 `status` 字段是否为 `"connected"`（连接失败要在 agent 开工前发现；默认连接超时 60 秒）。

### 12.3 认证

stdio server 用 `env` 字段传凭证；远程 server 用 `headers` 传 `Authorization`。MCP 规范支持 OAuth 2.1，但 **SDK 不代办 OAuth 流程**——应用自己完成 OAuth 拿到 access token，再放进 `headers`。

### 12.4 Tool search：工具多了怎么办

工具定义会吃 context（50 个 tool 约 10–20K token），且一次加载超过 30–50 个后模型选工具的准确率下降。**Tool search 默认开启**：工具定义不预载，agent 只拿到摘要，需要时搜索目录、每次加载 3–5 个最相关的进 context。

用 `env={"ENABLE_TOOL_SEARCH": ...}` 控制：

| 值 | 行为 |
|---|---|
| 未设置 | 开启（Vertex AI / 非第一方 `ANTHROPIC_BASE_URL` 时回退为全量预载） |
| `"true"` / `"false"` | 强制开 / 关 |
| `"auto"` | 工具定义超过 context 的 10% 才激活 |
| `"auto:5"` | 自定义阈值为 5% |

工具少于约 10 个时全量预载反而更快。可发现性靠 name 和 description 的关键词质量（`search_slack_messages` 优于 `query_slack`）。


In [ ]:
import os
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    SystemMessage,
    AssistantMessage,
    ResultMessage,
)


async def demo_external_mcp():
    options = ClaudeAgentOptions(
        mcp_servers={
            "playwright": {"command": "npx", "args": ["@playwright/mcp@latest"]}
        },
        allowed_tools=["mcp__playwright__*"],  # 不授权的话 Claude 看得到但调不了
    )
    async for message in query(
        prompt="Open example.com and describe what you see", options=options
    ):
        # init 消息里核对 MCP server 连接状态
        if isinstance(message, SystemMessage) and message.subtype == "init":
            print("MCP servers:", message.data.get("mcp_servers"))
        # 观察 Claude 实际调了哪些 MCP tool
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if hasattr(block, "name") and block.name.startswith("mcp__"):
                    print("MCP tool called:", block.name)
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_external_mcp()

## 13. 会话：continue / resume / fork 与持久化

Session = SDK 在 agent 工作期间累积的对话历史（prompt、每次 tool call、tool result、每个 response），自动写盘。它持久化的是**对话**，不是文件系统——要回滚 agent 的文件改动得用 file checkpointing（独立功能，§21）。

### 13.1 按应用形态选方案

| 你在构建 | 用什么 |
|---|---|
| 一次性任务，无 follow-up | 单次 `query()`，什么都不用管 |
| 单进程内多轮对话 | `ClaudeSDKClient`（自动跟踪 session，无需管 ID） |
| 进程重启后接着上次 | `continue_conversation=True`（接当前目录最近一次） |
| 恢复特定历史 session | 捕获 session ID，传 `resume` |
| 尝试另一条路又不丢原线 | `resume` + `fork_session=True` |
| 多主机 / serverless 部署 | `session_store` 外部存储 adapter |

三种延续方式的语义：

- **continue**：接当前目录里最近的 session，不用跟踪任何 ID，适合单对话应用。
- **resume**：接指定 session ID。多用户/多 session 场景必须用它。session ID 从 `ResultMessage.session_id` 拿（每个 result 都有，成功失败都在，所以触限后也能 resume 放宽限制继续）。
- **fork**：`resume=<id>` + `fork_session=True` 新建一个 session，起点是原历史的副本；原 session 不动，得到两条可独立 resume 的线。**fork 分支的是对话，不是文件系统**——fork 出的 agent 改了文件，所有 session 都看得到。

### 13.2 存储位置与跨主机

Session 文件存于 `~/.claude/projects/<encoded-cwd>/*.jsonl`；`<encoded-cwd>` 是绝对工作目录路径把非字母数字字符全部替换成 `-`（`/Users/me/proj` → `-Users-me-proj`）。设了 `CLAUDE_CONFIG_DIR` 则挪到该目录下。

> [!warning] resume 出来是全新会话？
> 最常见原因是 `cwd` 不一致——session 文件按工作目录归档，从别的目录 resume 会找不到文件。跨主机（CI、容器、serverless）三条路：① 把 `.jsonl` 搬到新主机**相同路径**且 `cwd` 一致；② 不搬 transcript，把结论存成应用状态塞进新 session 的 prompt（官方认为通常更稳）；③ 用 `session_store`。

### 13.3 `session_store`：镜像到外部存储

`ClaudeAgentOptions(session_store=<adapter>)` 把 transcript **镜像**（不是替代——本地盘永远先写）到 S3/Redis/数据库。adapter 实现 `SessionStore` 协议：必选 `append(key, entries)` / `load(key)`，可选 `list_sessions` / `delete` / `list_subkeys`。SDK 内置 `InMemorySessionStore` 供开发测试；`claude_agent_sdk.testing.run_session_store_conformance` 提供一致性测试。

镜像写是 best-effort：失败重试最多 2 次，仍失败则发 `mirror_error` system 消息、丢弃该批、agent 继续跑。**重试可能重复投递，adapter 的 `append` 要按 `entry.uuid` 去重。** SDK 从不主动清理 store 数据，保留策略（TTL/lifecycle）由 adapter 自己负责。

磁盘 session 的管理函数：`list_sessions()` / `get_session_messages()` / `get_session_info()` / `rename_session()` / `tag_session()`，可用来构建 session 选择器、transcript 查看器。（TypeScript 有 `persistSession: false` 可不落盘，Python 始终写盘。）


In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage


async def demo_sessions():
    session_id = None
    # 第一段：分析任务，捕获 session_id
    async for m in query(
        prompt="Read the authentication module",
        options=ClaudeAgentOptions(allowed_tools=["Read", "Glob"]),
    ):
        if isinstance(m, ResultMessage):
            session_id = m.session_id  # 成功失败都有，触限后也能 resume

    # resume：同一上下文继续，"it" 指代上一轮的 auth 模块
    async for m in query(
        prompt="Now find all places that call it",
        options=ClaudeAgentOptions(resume=session_id),
    ):
        if isinstance(m, ResultMessage) and m.subtype == "success":
            print(m.result)

    # fork：从同一起点分叉探索另一条路线，原 session 不受影响
    async for m in query(
        prompt="Instead, outline how OAuth2 would work for this module",
        options=ClaudeAgentOptions(resume=session_id, fork_session=True, max_turns=5),
    ):
        if isinstance(m, ResultMessage):
            print("forked session:", m.session_id)  # 与 session_id 不同


await demo_sessions()

## 14. 子 Agent：把专项任务外包

主 agent 可以派生**专职子 agent** 做聚焦子任务。
三大价值：
- **context 隔离**（中间 tool call/result 留在 subagent 里，只有最终消息回到父级——父级 context 只增长一份摘要）；
- **并行**（多个 subagent 并发，总耗时约等于最慢者）；
- **专用指令 + 收窄工具集**；

子 agent 通过内建的 `Agent` tool 调用，所以要把 `"Agent"` 加进 `allowed_tools` 才能免审批派发。Claude 依据每个子 agent 的 `description` 自主决定是否委派；prompt 里点名（"Use the code-reviewer agent to..."）可强制调用。

### 14.1 `AgentDefinition` 字段

| 字段 | 类型 | 说明 |
|---|---|---|
| `description` | `str`（必填） | 何时用这个 agent——Claude 靠它决定委派 |
| `prompt` | `str`（必填） | 该 agent 的专属 system prompt |
| `tools` | `list[str]` | 允许的工具；省略则继承父级全部 |
| `disallowedTools` | `list[str]` | 移除工具；支持 `mcp__server__*` / `mcp__*` 模式 |
| `model` | `str` | 模型覆盖：`'fable'`/`'opus'`/`'sonnet'`/`'haiku'`/`'inherit'` 或完整 model ID |
| `effort` | `str` | 按 agent 覆盖推理深度 |
| `permissionMode` | `str` | 该 agent 内的权限模式（父级 `bypassPermissions`/`acceptEdits` 时会被继承且不可覆盖） |
| `maxTurns` / `background` / `skills` / `memory` / `mcpServers` / `initialPrompt` | — | 轮次上限 / 后台运行 / 预载技能 / memory 来源 / 专属 MCP / 主线程首条输入 |

> [!warning] Python 命名坑
> `AgentDefinition` 的多词字段**保持 camelCase**（`disallowedTools`、`mcpServers`、`maxTurns`、`permissionMode`），不遵循 snake_case——它们直接映射 wire format。

### 14.2 子 agent 的信息边界

父级 → 子 agent 的**唯一通道**是 `Agent` tool 的 prompt 字符串，所以文件路径、报错、已做的决策必须显式写进那个 prompt。

| 子 agent 拿得到 | 拿不到 |
|---|---|
| 自己的 system prompt + Agent tool 的 prompt | 父级对话历史与 tool result |
| 项目 CLAUDE.md（经 `setting_sources`） | 父级的 system prompt |
| 继承或收窄后的工具集 | 未列进 `skills` 的技能全文 |

识别消息归属：`parent_tool_use_id` 是 SDK 附加在消息流对象上的归属元数据，不出现在给 LLM 的输入或 LLM 的响应里。源自子 agent loop 的 `AssistantMessage`（子 agent 的模型输出）和 `UserMessage`（子 agent loop 内的 tool result）带此字段，值为派生该子 agent 的那次 `Agent` tool 调用的 `ToolUseBlock.id`；主对话消息该字段为 `None`。检测派发看 `ToolUseBlock.name`——当前版本发 `"Agent"`，旧版本发 `"Task"`，**两个名字都要匹配**。

### 14.3 行为要点（版本相关）

- 子 agent 默认**后台运行**（v2.1.198+）；`background=True` 可强制常驻后台。
- 子 agent 可以再派子 agent，最深 5 层；不想让它继续派，就别给它 `Agent` tool。
- 子 agent transcript 独立落盘：不受主对话 compaction 影响、可跨重启 resume（从 Agent tool result 里解析 `agentId: <id>` 文本，resume 同一 session 后在 prompt 里点名恢复；内置 `Explore`/`Plan` 是 one-shot，不能恢复）。
- 权限提示成倍增加时：子 agent 不自动继承父级的临时批准，用 `PreToolUse` hook 或权限规则统一放行。
- 超大规模编排（几十到几百个 agent）用 `Workflow` tool，目前仅 TypeScript SDK 提供。

### 14.4 父子共享 state 的四条通道

context 隔离是特性不是缺陷——父子之间没有共享对话状态的机制，"共用 state"必须走对话之外的通道。按 state 的性质选：

| state 性质 | 通道 | 说明 |
|---|---|---|
| 静态共识（项目约定、术语） | CLAUDE.md | 双方都加载，每次请求重新注入，不受 compaction 影响 |
| 派发时的一次性输入（路径、报错、决策） | `Agent` tool 的 prompt | 软约束：主 agent 的 system prompt 里要求派发时带上；硬约束：`PreToolUse` hook 拦 `Agent` tool，用 `updatedInput` 把 state 强制拼进派发 prompt |
| 动态可变 state（进度、共享结论、计数器） | **进程内 MCP tool（首选）** | handler 闭包引用同一个 Python 对象，父子都调同一组读写 tool；应用进程是单一事实源，可加锁、校验、审计 |
| 只需父侧聚合、子 agent 不用读 | `SubagentStart` / `SubagentStop` hook | 在应用进程收集各子 agent 结果，不占任何 agent 的 context |

进程内 MCP tool 当"共享内存"的骨架：

```python
shared_state = {"findings": []}          # 应用进程里的单一事实源

@tool("get_state", "Read shared findings", {})
async def get_state(args):
    return {"content": [{"type": "text", "text": str(shared_state["findings"])}]}

@tool("add_finding", "Append a finding to shared state", {"finding": str})
async def add_finding(args):
    shared_state["findings"].append(args["finding"])
    return {"content": [{"type": "text", "text": "ok"}]}

state_server = create_sdk_mcp_server(name="state", version="1.0.0",
                                     tools=[get_state, add_finding])

options = ClaudeAgentOptions(
    mcp_servers={"state": state_server},
    allowed_tools=["Agent", "mcp__state__*"],
    agents={
        "worker": AgentDefinition(
            description="...", prompt="...",
            # 子 agent 的 tools 显式列上 state 工具（省略 tools 则继承全部，也能用）
            tools=["Read", "Grep", "mcp__state__get_state", "mcp__state__add_finding"],
        )
    },
)
```

备选是**文件系统黑板**：父子共享同一 `cwd`，state 写成 `state.json` 之类的文件，双方用 Read/Write 读写。零额外代码，但并行子 agent 写同一文件会竞态，且文件内容每次读取都消耗 context——留给"产物本身就是文件"的场景。

> [!warning] 两个边界
> fork 出的 session、并行的子 agent，隔离的都只是**对话**，文件系统始终共享——文件黑板天然可见，但也意味着写冲突要自己防。反过来，进程内 MCP tool 的 state 只活在应用进程里，进程重启即失；需要跨重启就把 tool 的存储换成 Redis/数据库。


In [ ]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AgentDefinition,
    ToolUseBlock,
    AssistantMessage,
    ResultMessage,
)


async def demo_subagents():
    async for message in query(
        prompt="Use the code-reviewer agent to review this codebase",
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Glob", "Grep", "Agent"],  # 注意含 Agent
            agents={
                "code-reviewer": AgentDefinition(
                    description="Expert code reviewer for quality and security reviews.",
                    prompt="Analyze code quality and suggest improvements.",
                    tools=["Read", "Glob", "Grep"],  # 只读收窄
                    model="sonnet",  # 子任务用小模型省成本
                )
            },
        ),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                # 新旧版本 tool 名不同，两个都匹配
                if isinstance(block, ToolUseBlock) and block.name in ("Agent", "Task"):
                    print("subagent invoked:", block.input.get("subagent_type"))
        # 源自子 agent loop 的消息对象带 parent_tool_use_id（主对话消息为 None）
        if getattr(message, "parent_tool_use_id", None):
            print("  (from inside subagent)")
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_subagents()

## 15. `ClaudeSDKClient`：多轮与打断

`query()` 每次新开会话；要在同一上下文里连续追问、或中途叫停，用 `ClaudeSDKClient`。

- `async with ClaudeSDKClient(options=...) as client`：推荐用法，退出时自动断开、清理子进程（也可手动 `connect()` / `disconnect()`）。
- `await client.query(prompt)` 发一轮，`async for m in client.receive_response()` 收这一轮的响应；再 `client.query(...)` 追问，上下文自动保留（client 内部持有 session ID）。
- `await client.interrupt()`：中途打断正在跑的任务（`query()` 做不到）。
- `await client.set_permission_mode(...)` / `await client.set_model(...)`：运行期动态切换权限模式 / 模型。
- MCP 连接管理：`get_mcp_status()` / `reconnect_mcp_server()` / `toggle_mcp_server()`。

典型形态是把它包成一个会话对象：外层 `while True` 收用户输入，特殊命令映射到 client 方法（`interrupt` → `client.interrupt()`，`new` → `disconnect()` 后重新 `connect()` 开新会话），普通输入走 `query()` + `receive_response()`。


In [3]:
import anyio
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
)


async def demo_multiturn():
    options = ClaudeAgentOptions(cwd=".", allowed_tools=["Read", "Glob"], max_turns=4)
    async with ClaudeSDKClient(options=options) as client:
        await client.query("Read the project and summarize the main modules.")
        async for m in client.receive_response():
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, TextBlock):
                        print("A:", b.text)

        # 追问，自动带上一轮上下文
        await client.query("Now focus only on authentication-related files.")
        async for m in client.receive_response():
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, TextBlock):
                        print("A:", b.text)


await demo_multiturn()

# anyio.run(demo_multiturn)

A: I'll explore the project structure to identify and summarize the main modules.
A: Let me read the Jupyter notebook to understand the project:
A: Based on my reading of this Jupyter notebook project about Claude Agent SDK for Python, here's a summary of the main modules:

## Project Overview
This is a comprehensive tutorial notebook titled **"Claude Agent SDK · Python 从 0 到 1"** (Claude Agent SDK Python from 0 to 1) that teaches how to use the Claude Agent SDK.

## Main Modules/Sections

### 1. **Core Concepts & Architecture** (Sections 1-2)
- Explains the fundamental difference between Client SDK and Agent SDK
- Details the agent loop mechanism and why Agent SDK exists
- Clarifies the relationship between Claude Code CLI, Agent SDK, Managed Agents, and Client SDK
- Describes the subprocess architecture and how the agent runtime works

### 2. **Entry Points & Configuration** (Sections 3-5)
- **Installation & Authentication**: Setup guide with API key configuration
- **Two Entry Point

## 16. 流式输入：用 async generator 驱动会话

`prompt` 传 async generator 就进入 **streaming input mode**（官方首选）：agent 作为长生命周期进程运行，逐条消费你 yield 的消息。相比单条字符串多出五个能力——消息内附图、多条消息排队顺序处理、实时打断、权限回调（§9.3 的前提）、多轮 context 自然保持。

消息 dict 格式固定：

```python
{"type": "user", "message": {"role": "user", "content": <字符串 或 content block 列表>}}
```

`content` 传列表时可混合文本与图片 block：

```python
{"type": "user", "message": {"role": "user", "content": [
    {"type": "text", "text": "Review this architecture diagram"},
    {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": image_base64}},
]}}
```

generator 里可以 `await`（等外部条件、等用户下一条输入），yield 之间的时间 agent 在处理已收到的消息——这就是构建 chat 界面的骨架。

> [!warning] Python 专属坑：generator 异常静默吞掉
> generator 内抛异常时，Python SDK 只在 debug 级别记日志，**session 静默卡住、不 raise**。流式会话挂住无输出时，先开 debug logging 检查自己的 generator。


In [ ]:
import asyncio
from claude_agent_sdk import (
    ClaudeSDKClient,
    ClaudeAgentOptions,
    AssistantMessage,
    TextBlock,
)


async def demo_streaming_input():
    async def message_generator():
        yield {
            "type": "user",
            "message": {"role": "user", "content": "Analyze this project structure"},
        }
        await asyncio.sleep(2)  # 模拟等待外部条件/用户输入
        yield {
            "type": "user",
            "message": {"role": "user", "content": "Now summarize it in one sentence"},
        }

    options = ClaudeAgentOptions(max_turns=10, allowed_tools=["Read", "Grep", "Glob"])
    async with ClaudeSDKClient(options) as client:
        await client.query(message_generator())  # 发送流式输入
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, TextBlock):
                        print(block.text)


await demo_streaming_input()

## 17. 流式输出：逐 token 拿增量

默认 SDK 攒完一整条响应才 yield `AssistantMessage`。要做打字机效果/实时 UI，设 `include_partial_messages=True`——流里会**额外**插入 `StreamEvent` 消息（常规 `AssistantMessage`/`ResultMessage` 照发不误）。

```python
@dataclass
class StreamEvent:
    uuid: str
    session_id: str
    event: dict[str, Any]         # 原始 Claude API stream event
    parent_tool_use_id: str | None  # 来自子 agent 时非空
```

两个要点：`StreamEvent` 直接从 `claude_agent_sdk` 顶层导入；`event` 是原始 API 事件 dict（不是累积文本，文本要自己拼），全部用 `.get()` 访问。

事件类型与到达顺序：

| 事件 | 含义 |
|---|---|
| `message_start` / `message_stop` | 一条消息开始 / 结束 |
| `content_block_start` / `content_block_stop` | 一个内容块开始 / 结束（text 或 tool_use） |
| `content_block_delta` | 增量：`delta.type == "text_delta"` 是文本片段（`delta["text"]`）；`"input_json_delta"` 是 tool 入参分片（`delta["partial_json"]`） |
| `message_delta` | 消息级更新（stop reason、usage） |

处理三步：判断消息是不是 `StreamEvent` → 取 `event["type"]` → 对 `content_block_delta` 再看 `delta` 类型。构建 UI 时用一个 `in_tool` 标志区分"正在执行 tool"（显示状态指示）和"正常输出文本"（直接打印 delta）。

限制：结构化输出（§18）不走 streaming delta，JSON 只出现在最终 `ResultMessage.structured_output`。


In [ ]:
import sys
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage, StreamEvent


async def demo_streaming_ui():
    options = ClaudeAgentOptions(
        include_partial_messages=True,
        allowed_tools=["Read", "Bash", "Grep"],
    )
    in_tool = False  # 区分"正在执行 tool"与"正常输出文本"

    async for message in query(
        prompt="Find all TODO comments in this directory", options=options
    ):
        if isinstance(message, StreamEvent):
            event = message.event
            event_type = event.get("type")

            if event_type == "content_block_start":
                block = event.get("content_block", {})
                if block.get("type") == "tool_use":
                    print(f"\n[Using {block.get('name')}...]", end="", flush=True)
                    in_tool = True
            elif event_type == "content_block_delta":
                delta = event.get("delta", {})
                if delta.get("type") == "text_delta" and not in_tool:
                    sys.stdout.write(delta.get("text", ""))  # 打字机效果
                    sys.stdout.flush()
            elif event_type == "content_block_stop":
                if in_tool:
                    print(" done", flush=True)
                    in_tool = False
        elif isinstance(message, ResultMessage):
            print("\n\n--- Complete ---")


await demo_streaming_ui()

## 18. 结构化输出：让 agent 返回验证过的 JSON

自由文本适合 chat，不适合程序消费。`output_format` 让 agent 随便用工具干活，**最终必须返回符合你 JSON Schema 的数据**——SDK 负责校验，不匹配就自动 re-prompt 重试；重试耗尽则以 `error_max_structured_output_retries` 收场而不是给你脏数据。

```python
options = ClaudeAgentOptions(
    output_format={"type": "json_schema", "schema": <JSON Schema dict>}
)
# 结果在 ResultMessage.structured_output（dict）
```

支持的 Schema 特性：全部基本类型、`enum`、`const`、`required`、嵌套 object、`$ref`。

**Pydantic 是 Python 侧的最佳拍档**：`Model.model_json_schema()` 生成 schema，`Model.model_validate(msg.structured_output)` 把结果还原成带类型的对象——定义、校验、消费三步全类型安全。

错误处理按 subtype 分流：

```python
if isinstance(message, ResultMessage):
    if message.subtype == "success" and message.structured_output:
        ...  # 用验证过的数据
    elif message.subtype == "error_max_structured_output_retries":
        ...  # 检查 message.errors 区分失败原因：schema 验证失败，还是 model fallback 撤回输出
```

避错三条官方建议：schema 保持聚焦（深嵌套 + 大量 required 更难满足）；任务可能拿不到的信息设为 optional 字段（如 git blame 的 author/date）；prompt 写清楚要输出什么。


In [ ]:
from pydantic import BaseModel
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage


class Step(BaseModel):
    step_number: int
    description: str
    estimated_complexity: str  # 'low' / 'medium' / 'high'


class FeaturePlan(BaseModel):
    feature_name: str
    summary: str
    steps: list[Step]
    risks: list[str]


async def demo_structured_output():
    async for message in query(
        prompt="Plan how to add dark mode support to a React app. Break it into implementation steps.",
        options=ClaudeAgentOptions(
            output_format={
                "type": "json_schema",
                "schema": FeaturePlan.model_json_schema(),  # Pydantic 生成 schema
            }
        ),
    ):
        if isinstance(message, ResultMessage):
            if message.subtype == "success" and message.structured_output:
                plan = FeaturePlan.model_validate(
                    message.structured_output
                )  # 还原为类型对象
                print(f"Feature: {plan.feature_name}")
                for step in plan.steps:
                    print(
                        f"{step.step_number}. [{step.estimated_complexity}] {step.description}"
                    )
            elif message.subtype == "error_max_structured_output_retries":
                print("Could not produce valid output:", message.errors)


await demo_structured_output()

## 19. 系统提示定制：四条路线

`system_prompt` 有三种形态，先记核心不对称：**SDK 默认的 system prompt 是"最小版"（只覆盖 tool calling），不是 CLI 的完整 Claude Code prompt**——而文件系统配置（§20）默认却是全加载。从 `claude -p` 迁移想要一致行为，必须显式用 preset。

| 写法 | 得到什么 |
|---|---|
| 不设 | 最小默认：只有 tool-calling 支持，无编码规范/回复风格/安全指令 |
| `system_prompt={"type": "preset", "preset": "claude_code"}` | CLI 同款完整 prompt（tool 指令、代码风格、语气、安全、环境上下文） |
| preset + `"append": "..."` | 全保留再追加自己的规则——**风险最低的定制**，做编码类产品首选 |
| `system_prompt="自定义字符串"` | 只有你写的内容；tool 指导和安全指令要自己补。适合不同 surface/identity/非编码 agent |

两个关键原理：

- **CLAUDE.md 不进 system prompt**——SDK 把它注入 conversation 作为项目上下文，与任何 system_prompt 配置兼容；它的加载由 `setting_sources` 控制，跟 preset 无关。
- **preset 里嵌了 per-session 动态段**（工作目录、平台、shell、git 状态），不同目录的两个 session 因此无法共享 prompt cache。设 `"exclude_dynamic_sections": True`（SDK ≥ 0.1.58，仅 preset 形式有效）把动态段挪到首条 user message，让相同配置跨用户跨机器命中同一条 cache。

```python
options = ClaudeAgentOptions(
    system_prompt={
        "type": "preset",
        "preset": "claude_code",
        "append": "Always include detailed docstrings and type hints in Python code.",
        "exclude_dynamic_sections": True,  # 跨 session 共享 prompt cache
    },
    setting_sources=["project"],  # CLAUDE.md 由这里控制，不由 preset 控制
)
```

第四条路线是 **output styles**（`~/.claude/output-styles/*.md` 或项目 `.claude/output-styles/`）：文件化的 system prompt 修改，默认**替换** preset 的软件工程指令，frontmatter 设 `keep-coding-instructions: true` 则叠加。Python SDK 没有编程选择 output style 的选项，只能靠 settings 文件激活——纯代码部署直接用 preset + append 替代。

四种方式的定位：CLAUDE.md 管**项目级持久约定**；output style 管**跨项目的角色化改写**；preset + append 管**session 级追加**；自定义字符串管**完全重造**（默认工具指导、安全指令、环境上下文三样全丢，须自行补齐）。


## 20. Claude Code 文件系统特性：setting_sources 是总开关

SDK 与 Claude Code 同底座，CLI 的文件系统配置（CLAUDE.md、rules、skills、hooks、settings、slash commands、output styles）SDK 都能加载，总开关是 `setting_sources`。

### 20.1 三个 source 的语义

**省略 `setting_sources` = 全加载（`["user", "project", "local"]`），与 CLI 行为一致；传 `[]` = 只认代码配置。**

| Source | 加载什么 | 位置要点 |
|---|---|---|
| `"project"` | 项目 CLAUDE.md、`.claude/rules/*.md`、项目 skills、项目 hooks、`.claude/settings.json`、`.mcp.json` | CLAUDE.md/rules 从 `<cwd>` 回溯**所有**父目录；skills 回溯到 repo root 为止；settings.json/hooks **只看** `<cwd>/.claude/` |
| `"user"` | `~/.claude/CLAUDE.md`、用户 rules/skills/settings | `~/.claude/` |
| `"local"` | `CLAUDE.local.md`、`.claude/settings.local.json` | 本机私有配置 |

各级 CLAUDE.md **叠加生效**，没有硬性优先级；指令冲突时结果取决于 Claude 的解读，官方建议写不冲突的规则。

> [!warning] setting_sources 管不住的四个输入（多租户必读)
> ① Managed policy settings（MDM/注册表/server-managed）始终加载；② `~/.claude.json` 全局配置始终读取（用 `env={"CLAUDE_CONFIG_DIR": ...}` 重定位）；③ Auto memory 会载入 system prompt（`CLAUDE_CODE_DISABLE_AUTO_MEMORY=1` 关闭）；④ claude.ai MCP connectors（订阅认证时）`mcp_servers={}` 压不掉。**多租户部署 = 每租户独立文件系统 + `setting_sources=[]` + 禁 auto memory**，不能只靠默认选项隔离。

### 20.2 Skills

Skill = `.claude/skills/<name>/SKILL.md`（YAML frontmatter + Markdown 正文），Claude 依据 `description` **自主调用**；SDK 没有编程注册 Skills 的 API（与 subagents 相反），只能走文件系统。启动时只发现 metadata，触发时才加载全文，所以对 context 很省。

`skills` 选项做会话内过滤：省略/`"all"` = 全部启用；`["pdf", "docx"]` = 只启用列出的；`[]` = 全禁。设了 `skills` 时 SDK 自动把 `Skill` tool 加进 allowed 名单，但若同时传了 `tools` 收窄名单，要自己把 `"Skill"` 加回去。

两个坑：`skills` 是 **context 过滤器不是沙箱**——没启用的 skill 文件还在磁盘上，Read/Bash 照样碰得到；SKILL.md 的 `allowed-tools` frontmatter **只在 CLI 生效**，SDK 下工具权限一律由主 `allowed_tools` 控制。

### 20.3 Slash commands

把 `/compact`、`/clear` 或自定义命令**直接当 prompt 字符串发**即可（这类命令是 SDK 输入，不是 CLI 专属）。可用命令列表在 init 消息 `message.data["slash_commands"]` 里。作用于对话历史的命令（`/compact`）要发到已有历史的会话（`continue_conversation=True` 或 streaming 模式）；压缩结果读 `compact_boundary` system 消息的 `data["compact_metadata"]`。自定义命令的 `.claude/commands/*.md` 是**遗留格式**，新命令推荐直接写成 skill（同样支持 `/name` 调用，还多了自主触发）。

### 20.4 Plugins

Plugin = 一包可跨项目分发的扩展（skills + agents + hooks + MCP servers）：

```python
options = ClaudeAgentOptions(
    plugins=[{"type": "local", "path": "./my-plugin"}]  # type 只接受 "local"
)
```

path 指向 plugin 根目录（`skills/`、`agents/`、`hooks/`、`.claude-plugin/` 的父目录）。plugin 的 skill/命令自动加 `plugin-name:` 前缀防冲突，调用发 `/plugin-name:skill-name`；加载验证读 init 消息的 `data["plugins"]` / `data["skills"]`。

### 20.5 特性选型速查

| 目标 | 用什么 |
|---|---|
| 项目约定 agent 一直遵守 | CLAUDE.md（`setting_sources` 含 `"project"`） |
| 按需加载的参考材料 / 可复用工作流 | Skills |
| 隔离子任务、新 context | Subagents（§14） |
| tool call 上的确定性逻辑 | Hooks（§11） |
| 接外部服务 | MCP（§12） |
| 打包分发以上全部 | Plugins |


## 21. 生产化：把 agent 变成可运维的服务

### 21.1 错误处理

SDK 层的异常类型（从 `claude_agent_sdk` 导入）：

```python
try:
    async for message in query(prompt="Hello"):
        print(message)
except CLINotFoundError:      # CLI 二进制找不到，重装 SDK
    ...
except ProcessError as e:     # 子进程非零退出，e.exit_code
    ...
except CLIJSONDecodeError:    # stdout 解析失败
    ...
```

再叠加两条已讲过的规则：业务级失败看 `ResultMessage.subtype`（§7.2）；`query()` 在 yield 错误 result 后还会 raise（§4.2）。超时与重试由 `env` 里的变量控制：`API_TIMEOUT_MS`（单请求超时，默认 600000）、`CLAUDE_CODE_MAX_RETRIES`（默认 10）、`CLAUDE_ASYNC_AGENT_STALL_TIMEOUT_MS`（后台子 agent 卡死看门狗）。

### 21.2 成本追踪

- **`total_cost_usd` 是客户端估算，不是账单**——SDK 用打包时内置的价格表本地计算，模型/价格变动时会漂移。开发洞察和粗略预算用它；对账用 Usage and Cost API。`max_budget_usd` 拿它做熔断。
- 三个统计口径：**step**（一次请求/响应，`AssistantMessage.usage` + `message_id`）、**query() 调用**（一个 `ResultMessage`，`total_cost_usd` 为该次调用累计）、**session**（多次 query() 串联，SDK 不提供合计，自己累加各次 result）。
- **并行 tool call 的多条 `AssistantMessage` 共享同一个 `message_id`、usage 相同**——按 step 统计要按 ID 去重，否则重复计数。
- 错误 result 同样带 `usage` / `total_cost_usd`（token 已经花了），失败也要记账。
- prompt cache 自动启用，无需配置；`usage` 里 `cache_creation_input_tokens`（写缓存，贵）和 `cache_read_input_tokens`（读缓存，便宜）分开看节省。短会话间隔超过 5 分钟导致 cache 反复过期时，设 `env={"ENABLE_PROMPT_CACHING_1H": "1"}` 换 1 小时 TTL（写入费率更高）。
- 按模型拆分：`ResultMessage.model_usage`（主 agent 用大模型、子 agent 用小模型时看 token 流向）。

### 21.3 可观测性：OpenTelemetry

CLI 内建 OTel 埋点（SDK 自己不产遥测，只透传配置），三种信号独立开关，导出到任何 OTLP 后端（Grafana、Datadog、Langfuse 等）：

```python
OTEL_ENV = {
    "CLAUDE_CODE_ENABLE_TELEMETRY": "1",
    "CLAUDE_CODE_ENHANCED_TELEMETRY_BETA": "1",   # 仅 traces 需要
    "OTEL_TRACES_EXPORTER": "otlp",
    "OTEL_METRICS_EXPORTER": "otlp",
    "OTEL_LOGS_EXPORTER": "otlp",
    "OTEL_EXPORTER_OTLP_PROTOCOL": "http/protobuf",
    "OTEL_EXPORTER_OTLP_ENDPOINT": "http://collector.example.com:4318",
}
options = ClaudeAgentOptions(env=OTEL_ENV)  # 或直接设在容器环境里
```

要点：**不要用 `console` exporter**（stdout 是 SDK 的消息通道）；span 结构是 `interaction → llm_request / tool → tool.execution`，带 `session.id` 属性可把多次 query() 串成一条时间线；应用侧已有活跃 span 时 SDK 自动注入 `TRACEPARENT`，agent trace 会挂在你的应用 trace 下；短生命周期进程把 `OTEL_*_EXPORT_INTERVAL` 调到 1000ms 防丢数据；多 agent 用 `OTEL_SERVICE_NAME` / `OTEL_RESOURCE_ATTRIBUTES` 区分，塞 `enduser.id` 可做逐用户审计。**prompt 与 tool 内容默认不导出**，`OTEL_LOG_USER_PROMPTS` / `OTEL_LOG_TOOL_DETAILS` 等是显式 opt-in。

### 21.4 文件快照与任务清单

**File checkpointing**：`enable_file_checkpointing=True` + `extra_args={"replay-user-messages": None}`（后者让 `UserMessage.uuid` 出现在流里作为 checkpoint ID）。`await client.rewind_files(checkpoint_id)` 把文件回滚到该点——**回滚的是磁盘不是对话**；只跟踪 `Write`/`Edit`/`NotebookEdit`（Bash 改的文件不算）；checkpoint 绑定 session；流迭代完后连接已关闭，要 resume 同一 session 发空 prompt 再调 rewind。

**任务清单**：agent 处理多步任务时会用 `TaskCreate` / `TaskUpdate` / `TaskGet` / `TaskList` 工具维护任务列表（v2.1.142 前是 `TodoWrite`），监听对应 `ToolUseBlock` 就能给用户渲染实时进度条。


### 21.5 Sandbox：给 Bash 套上笼子

`sandbox` 选项在 OS 层限制命令的文件系统与网络访问（Linux 依赖 `bubblewrap`/`socat`，macOS 用 `sandbox-exec`）：

```python
options = ClaudeAgentOptions(
    sandbox={
        "enabled": True,
        "autoAllowBashIfSandboxed": True,        # 沙箱内的 Bash 自动批准（默认 True）
        "network": {"allowedDomains": ["api.example.com"], "allowLocalBinding": True},
    }
)
```

关键语义（字段名 camelCase，直接映射 wire format）：

- `excludedCommands`（如 `["docker"]`）：静态名单，**总是**绕过沙箱，模型无权干预。
- `allowUnsandboxedCommands`（默认 True）：允许模型在 tool input 里设 `dangerouslyDisableSandbox: True` 申请出沙箱——这类请求**回落到权限系统**，`can_use_tool` 会被调用，可以在回调里做审计和白名单。**`bypassPermissions` + 该开关 = 模型可静默逃逸沙箱**，别组合使用。
- 沙箱不可用时默认降级为无沙箱执行（stderr 警告）；设 `"failIfUnavailable": True` 改为报 `error_during_execution`。
- 网络 allowlist 只按 hostname 过滤、不做 TLS 检查，domain fronting 可能绕过；更强保证要上 TLS 终止代理。

### 21.6 部署：子进程模型决定一切

一个 session = 一个 `claude` 子进程；三类状态默认落在容器本地盘（session transcript、CLAUDE.md、工作目录产物），**容器重启即丢**。由此推出四种 session 形态：

| 形态 | 做法 | 适用 |
|---|---|---|
| Ephemeral | 每任务一容器，跑完即毁 | 一次性任务（修 bug、抽取、转换） |
| Long-running | 常驻容器 + `ClaudeSDKClient` 挂长会话 | 邮件 agent、聊天机器人 |
| Hybrid | 临时容器 + `session_store` 启动时补水 | 间歇性回访的长项目（**store 是必需不是可选**） |
| Multi-agent | 一容器多子进程，各配独立 `cwd` | 多 agent 协作模拟 |

资源与扩缩：起步 1 GiB RAM / 5 GiB 盘 / 1 CPU 每 agent；并发上限由内存决定——`每主机 agent 数 = (主机 RAM - 开销) / 单 session 峰值 RSS`；长会话容器用 `sessionId` 一致性哈希钉到固定实例。已知限制：session 没有总超时（用 `max_turns` 兜底）、长会话内存增长（定期回收子进程）、宽扇出子 agent 会撞 API 限流（拆小批次）。

### 21.7 安全部署

威胁模型：prompt injection（agent 处理的内容里埋了指令）与模型失误。纵深防御四层：

1. **隔离**：sandbox runtime（轻量）→ 容器（`--cap-drop ALL` / `--read-only` / `--network none` + Unix socket 走代理）→ gVisor（拦截 syscall）→ microVM（Firecracker）。强度递增、开销递增。
2. **凭证走代理**：agent 环境里不放 key。`ANTHROPIC_BASE_URL` 指向代理由它注入 Anthropic key；其他服务的凭证要么走自定义 tool 转发到边界外执行，要么上 TLS 终止代理。
3. **最小权限文件系统**：代码只读挂载；**`.env`、`~/.aws/credentials`、`~/.ssh`、`*.pem` 等泄密文件挂载前剔除**；可写区用 tmpfs。
4. **审计**：代理记全部出站请求；OTel 的 `tool_decision` / `tool_result` 事件（§21.3）做逐用户审计流。

### 21.8 从 claude-code-sdk 迁移

旧包 `claude-code-sdk` → `claude-agent-sdk`，三处破坏性变化：包名与导入（`claude_code_sdk` → `claude_agent_sdk`）；`ClaudeCodeOptions` → `ClaudeAgentOptions`；**system prompt 不再默认 Claude Code 完整版**（要旧行为需显式 `system_prompt={"type": "preset", "preset": "claude_code"}`，见 §19）。`setting_sources` 默认值曾在 v0.1.0 短暂改为不加载、后已回退到全加载；Python ≤ 0.1.59 把 `setting_sources=[]` 当作省略处理，依赖空列表隔离前先升级。


## 22. 一套选型判断框架

**用 Client SDK**：只要一次性文本/结构化输出；已有成熟工具编排层，不想引高层抽象；要对每步工具调用做极细粒度控制。

**用 Agent SDK**：要 Claude 自己读代码/改代码/跑命令；做代码 agent、开发助手、自动修复、CI 机器人；不想手写复杂 tool loop。

**先用 Claude Code CLI**：只想快速验证任务可行性；还在调提示词、权限、流程；需要人工交互而非程序集成。CLI 与 SDK 同能力不同界面，工作流可直接互译——很多团队白天用 CLI 开发、生产跑 SDK。

**看向 Managed Agents**：要托管 sandbox 与会话基础设施、长时异步任务、不想自己运维。常见路径：本地用 Agent SDK 原型，生产再评估是否迁 Managed Agents。


## 23. 建议练习顺序

1. 跑 §5 `demo_query()`，只开 `Read`，先把消息流打印出来。
2. 跑 §7 `demo_read_stream()`，练**按块类型分流**——这是最该练熟的核心技能。
3. 把任务换成真实项目里的问题（"梳理认证流程""总结目录结构"）。
4. 跑 §15 `demo_multiturn()`，体会同上下文连续追问；再试 §13 的 resume / fork。
5. 跑 §10 `demo_custom_tool()`，理解自定义工具怎么接进来、`is_error` 如何保住循环。
6. 加 §9 的 `can_use_tool` 和 §11 的 Hook，把 agent 变"可控"。
7. 跑 §17 流式输出和 §18 结构化输出，这两个是接业务系统的关键形态。
8. 最后按 §21 把预算、sandbox、可观测配齐，模拟一次生产部署。

> 只读文档不跑代码，不会真正建立对 Agent SDK 的手感。至少跑通一次 `query()` 和一次 `ClaudeSDKClient`。

---

**术语速查**

- **tool loop / agent loop**：模型请求调工具→执行→回传结果→再请求，循环到收尾。Agent SDK 的核心是把这层内建。
- **turn**：循环里一个完整的"请求 tool → 执行 → 回灌"周期；`max_turns` 只计带 tool call 的 turn。
- **进程内 MCP（SDK MCP server）**：把本地 Python 函数当工具暴露给 agent，同进程运行，无需外部服务。
- **streaming input mode**：`prompt` 传 async generator 的输入模式，附图/排队/打断/权限回调的前提。
- **HITL（Human-in-the-Loop）**：agent 在关键点停下等真人批准/输入，靠 `can_use_tool` 挂起 + `AskUserQuestion` 实现。
- **compaction**：context 逼近上限时 SDK 自动把旧历史压缩成摘要；持久规则要放 CLAUDE.md。
- **setting_sources**：文件系统配置（CLAUDE.md/skills/hooks/settings）的加载总开关；省略=全加载。

**Python 专属坑速查**（正文均已展开，集中列一遍）：SDK 不自动加载 `.env`（§3）；`query()` 错误 result 后会 raise（§4）；`can_use_tool` 需流式输入 + `{"continue_": True}` dummy hook（§9.3）；被自动批准的调用到不了回调（§9.1）；流式输入 generator 异常静默卡死（§16）；自定义工具抛异常整个循环终止（§10.5）；进程内 server 不支持 `structuredContent`（§10.5）；`AgentDefinition` 多词字段是 camelCase（§14.1）；`SystemMessage` 的数据都在 `.data` dict 里（§7.1）；Python 无 `SessionStart`/`SessionEnd` 回调 hook（§11.1）、无 `auto` 权限模式（§9.2）、无 `persistSession: false`（§13.3）。
